# Combining & Reshaping DataFrames in Pandas

This notebook covers four essential operations for combining and reshaping pandas DataFrames:

1. **`merge()`** — SQL-style joins (inner, left, right, outer)
2. **`concat()`** — Stacking DataFrames vertically or horizontally
3. **`melt()`** — Wide → Long format transformation
4. **`pivot()` / `pivot_table()`** — Long → Wide format transformation

We use a student scores dataset throughout for consistency.

In [ ]:
import pandas as pd
import numpy as np

## Sample DataFrames

We have two DataFrames:
- `students` — student info (ID, Name, Age)
- `scores` — subject scores (StudentID, Subject, Score)

Note: StudentID=4 exists only in `students`, and StudentID=5 only in `scores` — useful for demonstrating join types.

In [ ]:
students = pd.DataFrame({
    "StudentID": [1, 2, 3, 4],
    "Name": ["Alice", "Bob", "Charlie", "David"],
    "Age": [20, 22, 19, 21]
})

scores = pd.DataFrame({
    "StudentID": [1, 2, 3, 5],
    "Subject": ["Math", "Science", "Math", "History"],
    "Score": [85, 90, 78, 92]
})

print("Students DataFrame:")
display(students)

print("\nScores DataFrame:")
display(scores)

---
## 1. `merge()` — Joining DataFrames

`pd.merge()` is the pandas equivalent of SQL JOINs. The `how` parameter controls which rows are kept:

| `how` | Behaviour |
|---|---|
| `inner` | Only rows with matching keys in **both** DataFrames |
| `left` | All rows from left, matching from right (NaN if no match) |
| `right` | All rows from right, matching from left |
| `outer` | All rows from both, NaN where no match |

In [ ]:
# Inner Join: only StudentIDs present in BOTH DataFrames (1, 2, 3)
merged_inner = pd.merge(students, scores, on="StudentID", how="inner")
print("Inner Join (StudentID 4 and 5 dropped):")
display(merged_inner)

In [ ]:
# Left Join: keep all students, NaN for StudentID=4 who has no score
merged_left = pd.merge(students, scores, on="StudentID", how="left")
print("Left Join (David has NaN score — no matching score record):")
display(merged_left)

In [ ]:
# Outer Join: keep all rows from both, NaN where no match
merged_outer = pd.merge(students, scores, on="StudentID", how="outer")
print("Outer Join (all rows, NaN for mismatches):")
display(merged_outer)

In [ ]:
# Common pitfall: merging on mismatched key columns raises KeyError
try:
    pd.merge(students, scores, left_on="Name", right_on="Subject")
except KeyError as e:
    print(f"KeyError: {e}")
    print("Tip: Use left_on / right_on when key column names differ between DataFrames.")

---
## 2. `concat()` — Stacking DataFrames

`pd.concat()` stacks DataFrames either:
- **Vertically** (`axis=0`) — add more rows (same columns)
- **Horizontally** (`axis=1`) — add more columns (same rows)

Unlike `merge()`, `concat()` does **not** align on key columns — it stacks by position.

In [ ]:
# Vertical concat: stack two identical DataFrames
# ignore_index=True resets the index to 0,1,2...
concat_vertical = pd.concat([students, students], axis=0, ignore_index=True)
print("Vertical Concat (8 rows, reset index):")
display(concat_vertical)

In [ ]:
# Horizontal concat with keys creates a MultiIndex column header
students_short = students.head(3)
scores_short = scores.head(3)

concat_horizontal = pd.concat(
    {"Students": students_short, "Scores": scores_short},
    axis=1
)
print("Horizontal Concat (side-by-side, MultiIndex columns):")
display(concat_horizontal)

In [ ]:
# merge() vs concat() — key distinction
# concat does NOT align by key — it aligns by index position
# This can silently produce wrong results if indexes don't match
df_a = pd.DataFrame({"A": [1, 2]}, index=[0, 1])
df_b = pd.DataFrame({"B": [10, 20]}, index=[1, 2])  # different index!

print("concat with mismatched index → NaN filling:")
display(pd.concat([df_a, df_b], axis=1))

---
## 3. `melt()` — Wide → Long Format

`melt()` unpivots a DataFrame from **wide format** (each subject in its own column) to **long format** (one row per observation). This is the format required by seaborn, plotly, and most ML libraries.

In [ ]:
# Wide format: each subject is a separate column
wide_df = pd.DataFrame({
    "StudentID": [1, 2, 3],
    "Math": [85, 90, 78],
    "Science": [88, 92, 80],
    "History": [75, 85, 70]
})
print("Wide Format:")
display(wide_df)

In [ ]:
# melt() converts to long format
# id_vars: columns to keep as-is (identifier columns)
# value_vars: columns to unpivot (defaults to all remaining)
melted = pd.melt(
    wide_df,
    id_vars=["StudentID"],
    value_vars=["Math", "Science", "History"],
    var_name="Subject",
    value_name="Score"
)
print("Long Format after melt():")
display(melted.sort_values("StudentID").reset_index(drop=True))

In [ ]:
# Common pitfall: specifying a value_vars column that doesn't exist
try:
    pd.melt(wide_df, id_vars=["StudentID"], value_vars=["English"])
except KeyError as e:
    print(f"KeyError: {e}")
    print("Tip: All columns listed in value_vars must exist in the DataFrame.")

---
## 4. `pivot()` and `pivot_table()` — Long → Wide Format

`pivot()` is the inverse of `melt()` — it reshapes long format back to wide. `pivot_table()` is its more powerful cousin that handles **duplicate entries** using an aggregation function.

In [ ]:
# Long format DataFrame
long_df = pd.DataFrame({
    "StudentID": [1, 1, 2, 2, 3, 3],
    "Subject": ["Math", "Science", "Math", "Science", "Math", "Science"],
    "Score": [85, 88, 90, 92, 78, 80]
})
print("Long Format:")
display(long_df)

In [ ]:
# pivot(): strict — requires unique index/column combinations
pivoted = long_df.pivot(index="StudentID", columns="Subject", values="Score")
pivoted.columns.name = None  # clean up column header label
print("pivot() result (wide format):")
display(pivoted)

In [ ]:
# pivot_table(): handles duplicates with aggregation
# Useful when the same StudentID + Subject combination appears multiple times
pivot_table = pd.pivot_table(
    long_df,
    index="StudentID",
    columns="Subject",
    values="Score",
    aggfunc="mean",
    fill_value=0
)
pivot_table.columns.name = None
print("pivot_table() with mean aggregation:")
display(pivot_table)

In [ ]:
# pivot() raises ValueError on duplicate entries — use pivot_table() instead
long_df_dups = pd.DataFrame({
    "StudentID": [1, 1, 1],
    "Subject": ["Math", "Math", "Science"],  # Math appears twice for StudentID=1
    "Score": [85, 87, 88]
})

try:
    long_df_dups.pivot(index="StudentID", columns="Subject", values="Score")
except ValueError as e:
    print(f"ValueError: {e}")
    print("\nFix: Use pivot_table() with aggfunc to handle duplicates:")
    display(pd.pivot_table(long_df_dups, index="StudentID", columns="Subject",
                           values="Score", aggfunc="mean"))

---
## Quick Reference

| Operation | Function | Use Case |
|---|---|---|
| Join on key column | `pd.merge()` | Combining tables like SQL JOINs |
| Stack rows/columns | `pd.concat()` | Appending more data, no key alignment |
| Wide → Long | `pd.melt()` | Preparing data for plotting/ML |
| Long → Wide (unique) | `df.pivot()` | Reshaping when no duplicates |
| Long → Wide (with agg) | `pd.pivot_table()` | Reshaping with summarization |

**Key rule of thumb:**
- Use `merge()` when you need to **align on a key**
- Use `concat()` when you need to **stack** without key alignment
- Use `melt()` ↔ `pivot()` to switch between **wide** and **long** format